# Chapter 12: Ethics and Responsible AI
## AI for Networking and Security Engineers — Volume 1 (Final Chapter)

This notebook demonstrates all responsible AI components from Chapter 12:

1. **Bias Detection** — vendor dominance and single-solution bias checks
2. **Debiased Requests** — prompt engineering to counteract training data bias
3. **Audit Logging** — immutable SQLite-based AI decision audit trail
4. **Data Sanitizer** — redact secrets before sending configs to AI API
5. **HITL Gate Simulation** — Crawl-Walk-Run approval workflow
6. **Decision Matrix** — AI vs traditional automation suitability scoring
7. **Responsible AI Stack** — all components working together

> **Note**: Replace `YOUR_API_KEY` with your real Anthropic API key for live AI demos.

In [ ]:
!pip install anthropic -q
print('Dependencies installed.')

In [ ]:
import os

# Load API key: tries Colab secrets first, falls back to manual entry
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception:
    import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
    print('API key set via manual entry.')

---
## Demo 1: Bias Detection

AI recommendations can exhibit training data bias — over-representing one vendor
or providing a single solution without alternatives.

This demo detects both types and flags them.

In [ ]:
import re

VENDOR_PATTERNS = {
    'cisco':    r'\b(cisco|ios|nxos|nx-os|catalyst|nexus|asr|isr|csr|vss|stackwise)\b',
    'juniper':  r'\b(juniper|junos|srx|mx|ex|qfx)\b',
    'arista':   r'\b(arista|eos|cvx)\b',
    'nokia':    r'\b(nokia|sros|sr-os|7750)\b',
    'fortinet': r'\b(fortinet|fortigate|fortios)\b',
}


def detect_vendor_bias(text: str) -> dict:
    counts = {v: len(re.findall(p, text, re.I)) for v, p in VENDOR_PATTERNS.items() if re.search(p, text, re.I)}
    if not counts:
        return {"bias_detected": False, "note": "vendor_agnostic"}
    total = sum(counts.values())
    dominant = max(counts, key=counts.get)
    pct = counts[dominant] / total * 100
    return {
        "bias_detected": pct > 70 and total > 3,
        "dominant_vendor": dominant,
        "percentage": round(pct, 1),
        "distribution": counts
    }


def detect_single_solution(text: str) -> dict:
    signals = [r'alternatively', r'another option', r'you could also',
               r'different approach', r'option [12345]', r'approach [12345]', r'tradeoff']
    found = sum(1 for s in signals if re.search(s, text, re.I))
    return {"bias_detected": found == 0, "alternative_indicators": found}


# Test cases
recommendations = [
    {
        "label": "Biased (Cisco-heavy, single solution)",
        "text": """For high availability, use Cisco VSS with two Catalyst 9600 switches.
Configure VSS domain and enable StackWise. Cisco provides the best IOS features for this.
The Cisco solution gives you sub-second failover with Cisco NSF/SSO support."""
    },
    {
        "label": "Good (Multi-vendor, alternatives provided)",
        "text": """For high availability in datacenter switching, you have several options:

Option 1: Vendor-neutral LACP + VRRP/HSRP — works on Cisco, Arista, Juniper, Nokia.
Option 2: Cisco VSS or StackWise — Cisco-specific, tight integration.
Option 3: Arista MLAG — alternatively, Juniper MC-LAG provides similar functionality.

The tradeoff is vendor lock-in vs simplicity. For new deployments, consider vendor-neutral first."""
    },
    {
        "label": "Vendor-agnostic (no specific products)",
        "text": """Use VRRP for default gateway redundancy and LACP for link aggregation.
These are open standards supported by all major vendors. Alternatively,
HSRP provides similar functionality with Cisco-specific enhancements."""
    }
]

print("Bias Detection Results")
print("=" * 60)

for rec in recommendations:
    print(f"\n📝 {rec['label']}")
    vb = detect_vendor_bias(rec['text'])
    sb = detect_single_solution(rec['text'])

    vendor_icon = "⚠️" if vb['bias_detected'] else "✅"
    solution_icon = "⚠️" if sb['bias_detected'] else "✅"

    print(f"  {vendor_icon} Vendor bias: {'DETECTED' if vb['bias_detected'] else 'None'}")
    if vb.get('distribution'):
        print(f"     Distribution: {vb['distribution']} | Dominant: {vb.get('dominant_vendor', 'N/A')} ({vb.get('percentage', 0)}%)")

    print(f"  {solution_icon} Single-solution bias: {'DETECTED' if sb['bias_detected'] else 'None'}")
    print(f"     Alternative indicators found: {sb['alternative_indicators']}")

print("\n✓ Bias detection demo complete!")

---
## Demo 2: Debiased Request — Get Multiple Options

When you detect bias, reframe the request with anti-bias instructions.
This demo shows the difference between a biased and debiased response.

In [ ]:
from anthropic import Anthropic

client = Anthropic()

question = "How should I configure high availability for WAN edge routers?"

# Standard request (may exhibit bias)
print("STANDARD REQUEST (may be biased):")
print("-" * 50)
std_resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=400,
    temperature=0,
    messages=[{"role": "user", "content": question}]
)
std_text = std_resp.content[0].text
print(std_text[:500])

# Check for bias
import re
VENDOR_PATTERNS = {
    'cisco': r'\b(cisco|ios|catalyst|asr|isr)\b',
    'juniper': r'\b(juniper|junos|srx|mx)\b',
    'arista': r'\b(arista|eos)\b',
}
counts = {v: len(re.findall(p, std_text, re.I)) for v, p in VENDOR_PATTERNS.items()}
print(f"\nVendor mentions in standard response: {counts}")

# Debiased request
print("\n" + "="*60)
print("DEBIASED REQUEST:")
print("-" * 50)

debiased_prompt = f"""{question}

Requirements:
1. Provide exactly 3 different approaches
2. Start with vendor-neutral/open-standard solutions
3. Include at least 2 different vendor examples if relevant
4. For each: Pros, Cons, Best for (deployment size/type)
5. Explicitly state tradeoffs"""

debiased_resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=600,
    temperature=0,
    messages=[{"role": "user", "content": debiased_prompt}]
)
debiased_text = debiased_resp.content[0].text
print(debiased_text[:600])

counts2 = {v: len(re.findall(p, debiased_text, re.I)) for v, p in VENDOR_PATTERNS.items()}
print(f"\nVendor mentions in debiased response: {counts2}")

print("\n✓ Debiased request demo complete!")

---
## Demo 3: Audit Logging — The AI Decision Database

Every AI call logged with: prompt hash, response preview, model, cost, outcome.
Uses SQLite for demo. In production: Splunk, CloudWatch Logs, or Kafka.

This is your 'show ai decision-database' command.

In [ ]:
import sqlite3, json, hashlib, time, uuid, os
from anthropic import Anthropic

client = Anthropic()
DB = "/tmp/ai_audit_demo.db"

# Initialize DB
conn = sqlite3.connect(DB)
conn.execute("""
    CREATE TABLE IF NOT EXISTS ai_audit (
        record_id TEXT PRIMARY KEY,
        timestamp TEXT,
        operation TEXT,
        device TEXT,
        model TEXT,
        prompt_hash TEXT,
        prompt_preview TEXT,
        response_preview TEXT,
        tokens_in INTEGER,
        tokens_out INTEGER,
        cost_usd REAL,
        risk_level TEXT,
        outcome TEXT DEFAULT 'pending'
    )
""")
conn.commit()
conn.close()


def ai_call_with_audit(
    prompt: str,
    operation: str,
    device: str = None,
    risk_level: str = "low",
    model: str = "claude-haiku-4-5-20251001",
    max_tokens: int = 500
) -> tuple:
    """Make AI call and automatically log it to audit DB."""
    resp = client.messages.create(
        model=model, max_tokens=max_tokens, temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    text = resp.content[0].text
    tin = resp.usage.input_tokens
    tout = resp.usage.output_tokens

    # Cost calculation (per-million-token rates: input, output)
    rates = {"claude-haiku-4-5-20251001": (1.0, 5.0), "claude-sonnet-4-20250514": (3.0, 15.0)}
    in_r, out_r = rates.get(model, (3.0, 15.0))
    cost = (tin / 1e6 * in_r) + (tout / 1e6 * out_r)

    record_id = str(uuid.uuid4())[:8]

    conn = sqlite3.connect(DB)
    conn.execute("""
        INSERT INTO ai_audit VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        record_id,
        time.strftime("%Y-%m-%d %H:%M:%S"),
        operation,
        device or "N/A",
        model,
        hashlib.sha256(prompt.encode()).hexdigest()[:8],
        prompt[:150],
        text[:300],
        tin, tout,
        round(cost, 6),
        risk_level,
        "pending"
    ))
    conn.commit()
    conn.close()

    return text, record_id


def record_decision(record_id: str, outcome: str, approver: str = "demo_user"):
    conn = sqlite3.connect(DB)
    conn.execute("UPDATE ai_audit SET outcome=? WHERE record_id=?", (outcome, record_id))
    conn.commit()
    conn.close()


def show_audit_log():
    conn = sqlite3.connect(DB)
    rows = conn.execute("""
        SELECT record_id, timestamp, operation, device, model, cost_usd, risk_level, outcome
        FROM ai_audit ORDER BY timestamp
    """).fetchall()
    conn.close()
    print(f"\n{'ID':^10} {'Time':^20} {'Operation':^20} {'Device':^15} {'Cost':^8} {'Risk':^8} {'Outcome':^15}")
    print("-" * 100)
    for r in rows:
        print(f"{r[0]:^10} {r[1]:^20} {r[2]:^20} {r[3]:^15} ${r[5]:^7.4f} {r[6]:^8} {r[7]:^15}")


def show_cost_summary():
    conn = sqlite3.connect(DB)
    rows = conn.execute("""
        SELECT model, COUNT(*), SUM(tokens_in), SUM(tokens_out), SUM(cost_usd)
        FROM ai_audit GROUP BY model
    """).fetchall()
    conn.close()
    print("\nCost Summary:")
    for r in rows:
        print(f"  {r[0]}: {r[1]} calls | {r[2]} in + {r[3]} out tokens | ${r[4]:.4f} total")


# Demo: make several audited AI calls
print("Audit Logging Demo")
print("=" * 55)

calls = [
    ("Security analysis: snmp-server community public RO — issues?", "security_scan", "edge-sw-01", "low"),
    ("Classify syslog: %BGP-5-ADJCHANGE: neighbor 10.1.1.2 Down", "log_classify", "core-rtr-01", "medium"),
    ("Should we remove ACL 150 from finance VLAN? Config: permit ip 203.0.0.0 0.255.255.255 10.0.1.0 0.0.0.255",
     "config_recommendation", "fin-rtr-01", "high"),
]

record_ids = []
for prompt, operation, device, risk in calls:
    text, rid = ai_call_with_audit(prompt, operation, device, risk)
    record_ids.append(rid)
    print(f"\n[{rid}] {operation} on {device} (risk={risk})")
    print(f"  Response: {text[:150]}...")

# Simulate human decisions on two of the three
record_decision(record_ids[0], "applied", "ops-team")
record_decision(record_ids[2], "rejected_by_human", "edy")
# record_ids[1] stays as 'pending'

print("\n" + "="*100)
print("AUDIT LOG ('show ai decision-database')")
show_audit_log()
show_cost_summary()
print("\n✓ Audit logging demo complete!")

---
## Demo 4: Data Sanitizer — Redact Secrets Before API

Never send passwords, SNMP strings, or authentication keys to a public API in plaintext.
This demo shows what gets redacted and what is safe to send.

In [ ]:
import re


SANITIZE_PATTERNS = [
    (r'(neighbor\s+[\d./]+\s+password\s+)\S+', r'\1[BGP_PASSWORD_REDACTED]', 'bgp_password'),
    (r'(enable\s+(?:secret|password)\s+(?:\d\s+)?)\S+', r'\1[PASSWORD_REDACTED]', 'enable_password'),
    (r'(username\s+\S+\s+(?:secret|password)\s+(?:\d\s+)?)\S+', r'\1[PASSWORD_REDACTED]', 'user_password'),
    (r'(snmp-server\s+community\s+)\S+(\s+(?:RO|RW))', r'\1[SNMP_COMMUNITY_REDACTED]\2', 'snmp_community'),
    (r'((?:radius|tacacs)[-\s]+server[^!]*key\s+)\S+', r'\1[RADIUS_KEY_REDACTED]', 'radius_key'),
    (r'(ntp\s+authentication-key\s+\d+\s+md5\s+)\S+', r'\1[NTP_KEY_REDACTED]', 'ntp_key'),
    (r'(ip\s+ospf\s+message-digest-key\s+\d+\s+md5\s+)\S+', r'\1[OSPF_KEY_REDACTED]', 'ospf_key'),
    (r'(pre-shared-key\s+(?:address\s+\S+\s+)?)\S+', r'\1[PSK_REDACTED]', 'preshared_key'),
    (r'\$9\$[A-Za-z0-9/]+', '[JUNIPER_ENCRYPTED_PASS_REDACTED]', 'juniper_pass'),
]


def sanitize_config(text: str) -> tuple:
    """Returns (sanitized_text, redacted_count, categories_found)"""
    sanitized = text
    total = 0
    found_categories = []

    for pattern, replacement, category in SANITIZE_PATTERNS:
        try:
            new_text, n = re.subn(pattern, replacement, sanitized, flags=re.IGNORECASE | re.DOTALL)
            if n > 0:
                sanitized = new_text
                total += n
                found_categories.append(f"{category} ({n}x)")
        except Exception:
            pass

    return sanitized, total, found_categories


# Raw config with multiple secrets
raw_config = """
hostname core-rtr-01
!
router bgp 65001
 neighbor 10.1.1.2 remote-as 65002
 neighbor 10.1.1.2 password Str0ngBGPp@ssword123!
 neighbor 10.1.1.6 remote-as 65003
 neighbor 10.1.1.6 password AnotherBGPsecret!
!
snmp-server community xKj9#SuperSecret RO
snmp-server community writeMe!Now RW
!
enable secret 5 $1$mERr$hx5rVt7rPNoS4wqbXKX7m0
!
username admin password 0 AdminP@ss2024!
username noc password 7 121A0C0411045D
!
ntp authentication-key 1 md5 NTPsecret123
!
ip ospf message-digest-key 1 md5 OSPFkey!2024
!
interface GigabitEthernet0/0
 description WAN Link to ISP
 ip address 203.0.113.1 255.255.255.252
!
interface GigabitEthernet0/1
 description Finance VLAN Uplink
 ip address 10.100.1.1 255.255.255.0
"""

sanitized, count, categories = sanitize_config(raw_config)

print("Data Sanitization Demo")
print("=" * 60)
print(f"\n⚠️  ORIGINAL CONFIG (DO NOT SEND TO AI API):")
print("-" * 50)
print(raw_config[:500])

print(f"\n✅ SANITIZED CONFIG (safe to send to AI API):")
print("-" * 50)
print(sanitized[:500])

print(f"\n📊 Sanitization Report:")
print(f"  Total values redacted: {count}")
print(f"  Categories found:")
for cat in categories:
    print(f"    - {cat}")

print("\nNote: IP addresses and network topology are preserved.")
print("The AI can analyze config structure without seeing actual secrets.")
print("\n✓ Data sanitizer demo complete!")

### Why Sanitization Is Non-Negotiable

When you send a config to an AI API, the request travels over the internet
to the provider's servers. If the config contains:

- **SNMP community strings** → anyone who intercepts them can read/write your devices
- **BGP passwords** → an attacker can hijack routing sessions
- **TACACS+/RADIUS keys** → full AAA bypass
- **Enable secrets** → complete device takeover

The sanitizer above replaces these with `***REDACTED***` before the config
leaves your environment. The AI can still analyze the structure — it just
cannot see the actual credentials.

> **Production tip:** In a real deployment, also sanitize SSH keys,
> API tokens, and JWT secrets. Add patterns for your organization's
> custom credential formats.

---
## Demo 5: HITL Gate — Crawl-Walk-Run Approval Workflow

The Human-in-the-Loop gate implements the Crawl-Walk-Run framework:
- CRAWL: All changes require human review
- WALK: All changes require human approval
- RUN: LOW risk auto-approved; MEDIUM/HIGH/CRITICAL require human

This simulated demo shows how different risk levels are handled in each phase.

In [ ]:
# Simulate the HITL gate (non-interactive version for Colab)

ALWAYS_BLOCK = ['reload', 'write erase', 'format ', 'erase startup-config', 'no router bgp']

def assess_risk_static(action: str) -> str:
    """Static risk assessment (no API call for demo)."""
    action_lower = action.lower()
    for blocked in ALWAYS_BLOCK:
        if blocked.lower() in action_lower:
            return "critical"
    if any(t in action_lower for t in ['show ', 'display ', 'debug ', 'ping ', 'traceroute ']):
        return "low"
    if any(t in action_lower for t in ['description', 'logging', 'ntp server', 'ip name']):
        return "low"
    if any(t in action_lower for t in ['interface', 'access-list', 'ip route', 'acl']):
        return "medium"
    if any(t in action_lower for t in ['router bgp', 'router ospf', 'neighbor', 'vlan', 'no shutdown']):
        return "high"
    return "medium"


def simulate_hitl_gate(action: str, phase: str, device: str) -> dict:
    """Simulate HITL gate decision (non-interactive for Colab)."""
    risk = assess_risk_static(action)

    # Hard block?
    for blocked in ALWAYS_BLOCK:
        if blocked.lower() in action.lower():
            return {"approved": False, "method": "auto_blocked", "risk": risk,
                    "reason": f"Safety block: contains '{blocked}'"}

    # Can auto-approve?
    auto_approve = False
    if phase == "run" and risk == "low":
        auto_approve = True

    if auto_approve:
        return {"approved": True, "method": "auto_approved", "risk": risk,
                "reason": f"Auto-approved: {risk} risk in {phase} phase"}
    else:
        # Simulate human approval based on risk
        human_approves = risk in ("low", "medium")  # In real life: user input
        return {
            "approved": human_approves,
            "method": "human_review_required",
            "risk": risk,
            "reason": "Human reviewed" if human_approves else "Human rejected",
            "approver": "edy" if human_approves else "N/A"
        }


test_changes = [
    ("show ip bgp summary",                          "core-rtr-01"),
    ("interface GigabitEthernet0/0\n description Updated by AI", "edge-sw-01"),
    ("router bgp 65001\n neighbor 10.1.1.2 soft-reset", "core-rtr-01"),
    ("reload in 5",                                  "prod-rtr-01"),
    ("ip route 0.0.0.0 0.0.0.0 203.0.113.254",      "edge-rtr-01"),
]

phases = ["crawl", "walk", "run"]

print("HITL Gate Simulation — Crawl/Walk/Run Framework")
print("=" * 70)

for phase in phases:
    print(f"\n{'─'*70}")
    print(f"Phase: {phase.upper()}")
    print(f"{'─'*70}")
    print(f"{'Action':^40} {'Risk':^8} {'Decision':^15} {'Method':^20}")
    print("-" * 85)

    for action, device in test_changes:
        result = simulate_hitl_gate(action, phase, device)
        icon = "✅" if result["approved"] else "🚫"
        decision = "APPROVED" if result["approved"] else "REJECTED/BLOCKED"
        print(f"{icon} {action[:38]:38} {result['risk']:^8} {decision:^15} {result['method']:^25}")

print("\n" + "="*70)
print("Key observations:")
print("  CRAWL: Nothing auto-approved — every recommendation goes to human")
print("  WALK : Nothing auto-approved — human approves all changes")
print("  RUN  : Only LOW risk auto-approved; everything else requires human")
print("  ALL  : CRITICAL (reload, write erase) always blocked, all phases")
print("\n✓ HITL gate demo complete!")

---
## Demo 6: Decision Matrix — AI vs Traditional Automation

Not every problem needs an AI solution.
This matrix scores tasks for AI suitability.
Score >= 60: USE_AI | 35-59: USE_HYBRID | < 35: USE_TRADITIONAL

In [ ]:
from dataclasses import dataclass


@dataclass
class TaskProfile:
    name: str
    is_deterministic: bool = False
    requires_interpretation: bool = False
    max_impact: str = "medium"  # low | medium | high | outage
    has_human_reviewer: bool = True
    contains_sensitive_data: bool = False
    is_regulated: bool = False
    has_reliable_alternative: bool = True


def score_task(t: TaskProfile) -> dict:
    score = 50
    reasons = []

    if t.requires_interpretation:    score += 25; reasons.append("✅ Requires interpretation")
    if not t.has_reliable_alternative: score += 15; reasons.append("✅ No reliable alternative")
    if not t.is_deterministic:       score += 10; reasons.append("✅ Non-deterministic task")
    if t.is_deterministic:           score -= 30; reasons.append("❌ Deterministic — script is better")
    if t.max_impact == "outage":     score -= 25; reasons.append("❌ Outage risk — needs deterministic")
    if not t.has_human_reviewer:     score -= 20; reasons.append("❌ No human reviewer")
    if t.contains_sensitive_data:    score -= 15; reasons.append("⚠️  Sensitive data — sanitize first")
    if t.is_regulated:               score -= 10; reasons.append("⚠️  Regulated — audit trail required")
    if t.max_impact == "high" and not t.has_human_reviewer:
        score -= 20; reasons.append("❌ High impact + no reviewer = too risky")

    rec = "USE_AI" if score >= 60 else "USE_HYBRID" if score >= 35 else "USE_TRADITIONAL"
    icons = {"USE_AI": "✅", "USE_HYBRID": "⚠️", "USE_TRADITIONAL": "❌"}
    return {"name": t.name, "score": score, "recommendation": rec, "icon": icons[rec], "reasons": reasons}


tasks = [
    TaskProfile("Syslog triage and severity classification",
                is_deterministic=False, requires_interpretation=True,
                max_impact="low", has_reliable_alternative=False),

    TaskProfile("BGP config change on production core router",
                is_deterministic=True, max_impact="outage",
                contains_sensitive_data=True, is_regulated=True),

    TaskProfile("Multi-vendor config security audit (400 devices)",
                is_deterministic=False, requires_interpretation=True,
                max_impact="medium", contains_sensitive_data=True,
                is_regulated=True, has_reliable_alternative=False),

    TaskProfile("SNMP threshold alert → page on-call engineer",
                is_deterministic=True, max_impact="low",
                has_human_reviewer=False),

    TaskProfile("Natural language troubleshooting assistant for NOC",
                is_deterministic=False, requires_interpretation=True,
                max_impact="low", has_human_reviewer=True,
                has_reliable_alternative=False),

    TaskProfile("Config backup rotation (copy run start, move to S3)",
                is_deterministic=True, max_impact="medium",
                has_human_reviewer=False),
]

print("AI Suitability Decision Matrix")
print("=" * 65)
print(f"{'Task':^40} {'Score':^8} {'Recommendation':^18}")
print("-" * 68)

for task in tasks:
    result = score_task(task)
    print(f"{result['icon']} {result['name'][:38]:38} {result['score']:^8} {result['recommendation']:^18}")

print("\nDetailed breakdown for ambiguous cases:")
for task in tasks:
    result = score_task(task)
    if result['recommendation'] == 'USE_HYBRID':
        print(f"\n⚠️  HYBRID: {result['name']}")
        for r in result['reasons']:
            print(f"   {r}")

print("\n✓ Decision matrix demo complete!")

---
## Demo 7: Full Responsible AI Stack

All components working together in one pipeline:
Sanitize → Call AI → Log audit → Check bias → HITL gate → Record decision

In [ ]:
from anthropic import Anthropic
import re, json, sqlite3, hashlib, time, uuid

client = Anthropic()


def run_responsible_ai_pipeline(
    raw_config: str,
    device: str,
    phase: str = "walk",
    operator: str = "demo_operator"
) -> dict:
    """
    Full responsible AI pipeline:
    𝟭: Sanitize sensitive data
    𝟮: Call AI for analysis
    𝟯: Detect bias in recommendation
    𝟰: Log audit record
    𝟱: Assess risk
    𝟲: Apply HITL gate
    𝟳: Record outcome
    """
    print(f"\n{'='*55}")
    print(f"Responsible AI Pipeline — {device}")
    print(f"Phase: {phase.upper()} | Operator: {operator}")
    print('='*55)

    # Step 1: Sanitize
    sanitize_patterns = [
        (r'(neighbor\s+[\d./]+\s+password\s+)\S+', r'\1[BGP_PASS_REDACTED]'),
        (r'(snmp-server\s+community\s+)\S+(\s+(?:RO|RW))', r'\1[SNMP_REDACTED]\2'),
        (r'(enable\s+(?:secret|password)\s+(?:\d\s+)?)\S+', r'\1[PASS_REDACTED]'),
        (r'(username\s+\S+\s+(?:secret|password)\s+(?:\d\s+)?)\S+', r'\1[PASS_REDACTED]'),
    ]
    sanitized = raw_config
    redacted = 0
    for pattern, replacement in sanitize_patterns:
        sanitized, n = re.subn(pattern, replacement, sanitized, flags=re.IGNORECASE)
        redacted += n
    print(f"\n𝟭 Sanitization: {redacted} sensitive values redacted")

    # Step 2: AI analysis
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        temperature=0,
        messages=[{"role": "user", "content": f"""
Analyze this Cisco IOS config for security issues and best practice violations.
Config: {sanitized[:2000]}
Return JSON: {{"risk_level": "low|medium|high|critical", "issues": [], "recommendations": []}}"""}]
    )
    ai_text = resp.content[0].text
    tin, tout = resp.usage.input_tokens, resp.usage.output_tokens
    cost = (tin / 1e6 * 1.0) + (tout / 1e6 * 5.0)
    print(f"𝟮 AI Analysis: {tin} in + {tout} out tokens | ${cost:.5f}")

    # Step 3: Bias check
    vendor_counts = {v: len(re.findall(p, ai_text, re.I)) for v, p in {
        'cisco': r'\b(cisco|ios|catalyst)\b',
        'juniper': r'\b(juniper|junos)\b',
        'arista': r'\b(arista|eos)\b'
    }.items() if re.search(p, ai_text, re.I)}
    bias_note = f"Vendor mentions: {vendor_counts}" if vendor_counts else "vendor-agnostic"
    print(f"𝟯 Bias Check: {bias_note}")

    # Step 4: Parse risk level
    risk_match = re.search(r'risk_level.*?(?:low|medium|high|critical)', ai_text, re.I)
    risk_level = "medium"
    if risk_match:
        risk_text = risk_match.group(0).lower()
        for r in ['critical', 'high', 'medium', 'low']:
            if r in risk_text:
                risk_level = r
                break
    print(f"𝟰 Risk Assessment: {risk_level.upper()}")

    # Step 5: HITL gate
    auto_approve = phase == "run" and risk_level == "low"
    if auto_approve:
        outcome = "applied"
        approver = "auto_system"
        print(f"𝟱 HITL Gate: AUTO-APPROVED (low risk, run phase)")
    elif risk_level == "critical":
        outcome = "auto_blocked"
        approver = "safety_system"
        print(f"𝟱 HITL Gate: AUTO-BLOCKED (critical risk)")
    else:
        # Simulate human approval for demo
        print(f"𝟱 HITL Gate: Human review required ({risk_level} risk, {phase} phase)")
        print(f"   [Demo: simulating human approval for medium/high risk in walk phase]")
        outcome = "applied"
        approver = operator

    print(f"𝟲 Outcome: {outcome.upper()} by {approver}")
    print(f"\nAI Recommendation (first 300 chars):")
    print(f"  {ai_text[:300]}...")

    return {
        "device": device,
        "risk_level": risk_level,
        "outcome": outcome,
        "approver": approver,
        "cost_usd": cost,
        "secrets_redacted": redacted
    }


# Run on two different configs
test_configs = [
    ("""
hostname edge-sw-01
snmp-server community public RO
line vty 0 4
 transport input telnet
 login
enable password cisco
""", "edge-sw-01"),
    ("""
hostname core-rtr-01
service password-encryption
ip ssh version 2
router bgp 65001
 neighbor 10.1.1.2 remote-as 65002
 neighbor 10.1.1.2 password Str0ngPass!
line vty 0 4
 login local
 transport input ssh
""", "core-rtr-01")
]

results = []
for config, device in test_configs:
    result = run_responsible_ai_pipeline(config, device, phase="walk")
    results.append(result)

print(f"\n{'='*55}")
print("Pipeline Summary:")
total_cost = sum(r['cost_usd'] for r in results)
total_redacted = sum(r['secrets_redacted'] for r in results)
print(f"  Devices processed    : {len(results)}")
print(f"  Secrets redacted     : {total_redacted}")
print(f"  Total cost           : ${total_cost:.5f}")
print(f"  All changes approved : {all(r['outcome'] == 'applied' for r in results)}")
print("\n✓ Responsible AI stack demo complete!")

---
## Summary — Volume 1 Complete

| Component | What It Does | Chapter Ref |
|-----------|--------------|-------------|
| Bias Detector | Flags vendor dominance and single-solution responses | Ch 12 |
| Data Sanitizer | Redacts secrets before sending to AI API | Ch 12 |
| Audit Logger | Immutable record of every AI decision | Ch 12 |
| HITL Gate | Crawl/Walk/Run approval framework | Ch 12 |
| Decision Matrix | AI vs traditional automation scoring | Ch 12 |

**The responsible AI stack wraps everything you built in Chapters 1-11.**

**Volume 1: Foundations — COMPLETE**

Volume 2 covers: RAG pipelines, LangGraph agents, multi-agent NOC systems, and self-healing networks.